# MichSet Audio Extractor for Google Colab

This notebook extracts actual audio files from MichSet Arrow datasets and prepares them for TTS training.

**What this does:**
1. Installs `torchcodec` for audio processing
2. Extracts audio files from Arrow format datasets
3. Creates proper file structure for TTS training
4. Verifies audio quality and duration

**Prerequisites:**
- Google Colab with GPU (recommended)
- Google Drive mounted with voice-lab-data
- MichSet datasets already downloaded

## 1. Environment Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
from pathlib import Path

# Set up paths
BASE_DIR = Path('/content/drive/MyDrive/voice-lab-data')
MICHSET_DIR = BASE_DIR / 'raw' / 'michsethowusu'
AUDIO_OUTPUT_DIR = BASE_DIR / 'processed' / 'michset_audio_extracted'
AUDIO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base directory: {BASE_DIR}")
print(f"MichSet datasets: {MICHSET_DIR}")
print(f"Audio output: {AUDIO_OUTPUT_DIR}")
print(f"MichSet exists: {MICHSET_DIR.exists()}")

## 2. Install Dependencies

In [ ]:
# Install torchcodec and other required packages
!pip install -q torch torchaudio
!pip install -q datasets
!pip install -q librosa soundfile
!pip install -q tqdm pandas

# Install torchcodec - this is the key package we need
print("Installing torchcodec...")
!pip install -q torchcodec

# Verify installation
try:
    import torchcodec
    print(f"✅ torchcodec installed successfully: {torchcodec.__version__}")
except ImportError:
    print("❌ torchcodec installation failed")
    print("Trying alternative installation...")
    !conda install -y -c conda-forge torchcodec

# Import other packages
import torch
import torchaudio
import librosa
import soundfile as sf
import pandas as pd
import numpy as np
from datasets import load_from_disk
from tqdm.auto import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
print("Dependencies installed successfully!")

## 3. Audio Extraction Class

In [ ]:
class MichSetAudioExtractor:
    """Extract audio files from MichSet Arrow datasets with torchcodec."""
    
    def __init__(self, output_dir: Path, target_sr: int = 16000):
        self.output_dir = output_dir
        self.target_sr = target_sr
        self.extraction_stats = {}
        
    def extract_dataset_audio(self, dataset_path: Path, max_samples: int = None) -> dict:
        """Extract audio files from a single MichSet dataset."""
        
        dataset_name = dataset_path.name
        print(f"\n🎵 Processing {dataset_name}...")
        
        if not dataset_path.exists():
            print(f"❌ Dataset not found: {dataset_path}")
            return {'status': 'missing', 'extracted': 0}
        
        try:
            # Load dataset with audio decoding enabled
            print("  Loading dataset...")
            dataset = load_from_disk(str(dataset_path))
            
            if 'train' not in dataset:
                print("  ❌ No training split found")
                return {'status': 'no_train_split', 'extracted': 0}
            
            train_data = dataset['train']
            total_samples = len(train_data)
            
            # Limit samples if specified
            if max_samples:
                total_samples = min(total_samples, max_samples)
            
            print(f"  📊 Total samples: {len(train_data)}, Processing: {total_samples}")
            
            # Create output directory for this dataset
            dataset_output_dir = self.output_dir / dataset_name
            dataset_output_dir.mkdir(parents=True, exist_ok=True)
            
            # Extract audio files
            extracted_files = []
            failed_extractions = 0
            
            for i in tqdm(range(total_samples), desc=f"Extracting {dataset_name}"):
                try:
                    sample = train_data[i]
                    
                    # Get audio data and transcription
                    if 'audio' in sample:
                        # New format with embedded audio
                        audio_data = sample['audio']
                        if isinstance(audio_data, dict) and 'array' in audio_data:
                            audio_array = audio_data['array']
                            sample_rate = audio_data.get('sampling_rate', 16000)
                        else:
                            print(f"    ⚠️ Unexpected audio format at sample {i}")
                            continue
                    elif 'audio_path' in sample:
                        # Old format with audio paths - this should work now with torchcodec
                        print(f"    ℹ️ Sample {i} has audio_path, attempting to load...")
                        # Skip for now, focus on embedded audio
                        continue
                    else:
                        continue
                    
                    transcription = sample.get('transcription', '')
                    
                    if len(transcription.strip()) == 0:
                        continue  # Skip samples without transcription
                    
                    # Convert to numpy array if needed
                    if not isinstance(audio_array, np.ndarray):
                        audio_array = np.array(audio_array)
                    
                    # Resample if needed
                    if sample_rate != self.target_sr:
                        audio_array = librosa.resample(
                            audio_array, 
                            orig_sr=sample_rate, 
                            target_sr=self.target_sr
                        )
                    
                    # Quality checks
                    duration = len(audio_array) / self.target_sr
                    if duration < 0.5 or duration > 15.0:
                        continue  # Skip too short or too long samples
                    
                    if np.max(np.abs(audio_array)) < 0.01:
                        continue  # Skip silent samples
                    
                    # Save audio file
                    output_filename = f"{dataset_name}_{i:05d}.wav"
                    output_path = dataset_output_dir / output_filename
                    
                    sf.write(str(output_path), audio_array, self.target_sr)
                    
                    # Store metadata
                    extracted_files.append({
                        'audio_file': str(output_path),
                        'transcription': transcription,
                        'duration': duration,
                        'sample_rate': self.target_sr,
                        'original_index': i,
                        'dataset': dataset_name
                    })
                    
                except Exception as e:
                    failed_extractions += 1
                    if failed_extractions < 5:  # Only print first few errors
                        print(f"    ⚠️ Error extracting sample {i}: {str(e)[:50]}")
            
            # Save metadata
            if extracted_files:
                metadata_df = pd.DataFrame(extracted_files)
                metadata_path = dataset_output_dir / 'extracted_metadata.csv'
                metadata_df.to_csv(metadata_path, index=False)
                
                # Create TTS-ready metadata.txt
                tts_metadata = []
                for _, row in metadata_df.iterrows():
                    audio_file = row['audio_file']
                    text = str(row['transcription']).replace('|', '-')  # Remove pipes
                    speaker_id = f"{dataset_name}_speaker"
                    tts_metadata.append(f"{audio_file}|{text}|{speaker_id}")
                
                tts_metadata_path = dataset_output_dir / 'metadata.txt'
                with open(tts_metadata_path, 'w', encoding='utf-8') as f:
                    f.write('\n'.join(tts_metadata))
                
                print(f"  ✅ Extracted {len(extracted_files)} audio files")
                print(f"  📁 Saved to: {dataset_output_dir}")
                print(f"  📊 Average duration: {metadata_df['duration'].mean():.2f}s")
                print(f"  ⚠️ Failed extractions: {failed_extractions}")
            else:
                print(f"  ❌ No audio files extracted")
            
            return {
                'status': 'success',
                'extracted': len(extracted_files),
                'failed': failed_extractions,
                'output_dir': str(dataset_output_dir),
                'metadata': extracted_files[:5]  # First 5 for verification
            }
            
        except Exception as e:
            print(f"  ❌ Dataset processing failed: {str(e)}")
            return {'status': 'error', 'error': str(e), 'extracted': 0}

# Initialize extractor
extractor = MichSetAudioExtractor(AUDIO_OUTPUT_DIR, target_sr=16000)
print("Audio extractor initialized!")

## 4. Extract Audio from All MichSet Datasets

In [ ]:
# Define MichSet datasets to process
michset_datasets = [
    'twi_multispeaker_audio_transcribed',
    'akuapem_multispeaker_audio_transcribed', 
    'fante_multispeaker_audio_transcribed'
]

# Extraction settings
MAX_SAMPLES_PER_DATASET = 2000  # Limit for Colab resources

print(f"🚀 Starting audio extraction...")
print(f"📊 Max samples per dataset: {MAX_SAMPLES_PER_DATASET}")
print(f"🎯 Target sample rate: 16kHz")
print("=" * 60)

extraction_results = {}
total_extracted = 0

for dataset_name in michset_datasets:
    dataset_path = MICHSET_DIR / dataset_name
    
    result = extractor.extract_dataset_audio(
        dataset_path=dataset_path,
        max_samples=MAX_SAMPLES_PER_DATASET
    )
    
    extraction_results[dataset_name] = result
    total_extracted += result.get('extracted', 0)
    
    # Print summary for this dataset
    print(f"\n📋 {dataset_name} Summary:")
    print(f"  Status: {result['status']}")
    print(f"  Extracted: {result.get('extracted', 0)} files")
    if result.get('failed', 0) > 0:
        print(f"  Failed: {result['failed']} samples")

print("\n" + "=" * 60)
print(f"🎉 EXTRACTION COMPLETE!")
print(f"📊 Total audio files extracted: {total_extracted}")
print(f"📁 Output directory: {AUDIO_OUTPUT_DIR}")

## 5. Verify Extracted Audio

In [ ]:
# Verify extracted audio files
print("🔍 Verifying extracted audio files...")

verification_results = {}

for dataset_name in michset_datasets:
    dataset_dir = AUDIO_OUTPUT_DIR / dataset_name
    
    if not dataset_dir.exists():
        print(f"❌ {dataset_name}: Directory not found")
        continue
    
    # Count audio files
    wav_files = list(dataset_dir.glob('*.wav'))
    metadata_file = dataset_dir / 'extracted_metadata.csv'
    tts_metadata_file = dataset_dir / 'metadata.txt'
    
    print(f"\n📁 {dataset_name}:")
    print(f"  Audio files: {len(wav_files)}")
    print(f"  Metadata CSV: {'✅' if metadata_file.exists() else '❌'}")
    print(f"  TTS metadata: {'✅' if tts_metadata_file.exists() else '❌'}")
    
    if len(wav_files) > 0:
        # Test first audio file
        test_file = wav_files[0]
        try:
            audio, sr = librosa.load(str(test_file), sr=16000)
            duration = len(audio) / sr
            print(f"  ✅ Sample audio: {duration:.2f}s, SR: {sr}Hz")
            
            # Load metadata and show sample
            if metadata_file.exists():
                df = pd.read_csv(metadata_file)
                sample_row = df.iloc[0]
                print(f"  📝 Sample text: {sample_row['transcription'][:50]}...")
                print(f"  📊 Avg duration: {df['duration'].mean():.2f}s")
                
                verification_results[dataset_name] = {
                    'audio_files': len(wav_files),
                    'avg_duration': df['duration'].mean(),
                    'total_duration': df['duration'].sum(),
                    'status': 'verified'
                }
                
        except Exception as e:
            print(f"  ❌ Error testing audio: {str(e)[:50]}")
            verification_results[dataset_name] = {'status': 'error', 'error': str(e)}
    else:
        verification_results[dataset_name] = {'status': 'no_files', 'audio_files': 0}

# Overall statistics
print("\n" + "=" * 60)
print("📊 VERIFICATION SUMMARY")
print("=" * 60)

total_files = sum(r.get('audio_files', 0) for r in verification_results.values())
total_hours = sum(r.get('total_duration', 0) for r in verification_results.values()) / 3600

print(f"Total extracted files: {total_files}")
print(f"Total audio duration: {total_hours:.2f} hours")

for dataset, results in verification_results.items():
    if results['status'] == 'verified':
        print(f"  {dataset}: {results['audio_files']} files, {results['total_duration']/60:.1f}min")

## 6. Create TTS-Ready Dataset Structure

In [ ]:
# Create TTS-ready dataset structure matching the notebook expectations
TTS_READY_DIR = BASE_DIR / 'processed' / 'tts_ready'
TTS_READY_DIR.mkdir(parents=True, exist_ok=True)

print("🎯 Creating TTS-ready dataset structure...")

# Map MichSet datasets to TTS dialect directories
dataset_mapping = {
    'twi_multispeaker_audio_transcribed': 'asante_twi',
    'akuapem_multispeaker_audio_transcribed': 'akuapem_twi',
    'fante_multispeaker_audio_transcribed': 'fante'
}

tts_datasets_created = {}

for michset_name, tts_dialect in dataset_mapping.items():
    source_dir = AUDIO_OUTPUT_DIR / michset_name
    target_dir = TTS_READY_DIR / tts_dialect
    
    if not source_dir.exists():
        print(f"⚠️ Source directory not found: {source_dir}")
        continue
    
    target_dir.mkdir(parents=True, exist_ok=True)
    
    # Copy metadata.txt to expected location
    source_metadata = source_dir / 'metadata.txt'
    target_metadata = target_dir / 'metadata.txt'
    
    if source_metadata.exists():
        # Copy and update paths in metadata
        with open(source_metadata, 'r', encoding='utf-8') as f:
            metadata_lines = f.readlines()
        
        print(f"📁 Creating {tts_dialect}: {len(metadata_lines)} samples")
        
        # Write metadata with original paths (audio files stay in extracted location)
        with open(target_metadata, 'w', encoding='utf-8') as f:
            f.writelines(metadata_lines)
        
        # Also copy CSV metadata
        source_csv = source_dir / 'extracted_metadata.csv'
        target_csv = target_dir / 'dataset.csv'
        
        if source_csv.exists():
            df = pd.read_csv(source_csv)
            df.to_csv(target_csv, index=False)
        
        tts_datasets_created[tts_dialect] = {
            'samples': len(metadata_lines),
            'metadata_file': str(target_metadata),
            'source_audio_dir': str(source_dir)
        }
        
        print(f"  ✅ {target_metadata}")
    else:
        print(f"❌ No metadata found for {michset_name}")

# Create combined dataset
print(f"\n🔗 Creating combined dataset...")

combined_dir = TTS_READY_DIR / 'combined_twi'
combined_dir.mkdir(parents=True, exist_ok=True)

all_metadata_lines = []
for tts_dialect, info in tts_datasets_created.items():
    metadata_file = Path(info['metadata_file'])
    if metadata_file.exists():
        with open(metadata_file, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            all_metadata_lines.extend(lines)

if all_metadata_lines:
    combined_metadata = combined_dir / 'metadata.txt'
    with open(combined_metadata, 'w', encoding='utf-8') as f:
        f.writelines(all_metadata_lines)
    
    print(f"  ✅ Combined dataset: {len(all_metadata_lines)} total samples")
    print(f"  📁 {combined_metadata}")

# Summary
print("\n" + "=" * 60)
print("🎉 TTS-READY DATASET CREATED!")
print("=" * 60)

print(f"📁 TTS datasets directory: {TTS_READY_DIR}")
print(f"📊 Datasets created:")

total_tts_samples = 0
for dialect, info in tts_datasets_created.items():
    samples = info['samples']
    total_tts_samples += samples
    print(f"  {dialect}: {samples} samples")

print(f"\n🎯 Total TTS training samples: {total_tts_samples}")
print(f"\n✅ Ready for 005_phase3_tts_development.ipynb!")
print(f"\n📝 Next steps:")
print(f"  1. Open 005_phase3_tts_development.ipynb")
print(f"  2. Run TTS training with real audio files")
print(f"  3. Test voice synthesis!")

## 7. Quick Audio Test

In [ ]:
# Test a few audio samples
from IPython.display import Audio, display
import matplotlib.pyplot as plt

print("🔊 Testing extracted audio samples...")

# Find first few audio files
test_files = []
for dataset_name in michset_datasets:
    dataset_dir = AUDIO_OUTPUT_DIR / dataset_name
    if dataset_dir.exists():
        wav_files = list(dataset_dir.glob('*.wav'))[:2]  # First 2 files
        test_files.extend(wav_files)

if test_files:
    for i, audio_file in enumerate(test_files[:4]):  # Test max 4 files
        try:
            # Load audio
            audio, sr = librosa.load(str(audio_file), sr=16000)
            duration = len(audio) / sr
            
            print(f"\n🎵 Sample {i+1}: {audio_file.name}")
            print(f"   Duration: {duration:.2f}s, Sample Rate: {sr}Hz")
            
            # Find corresponding transcription
            metadata_file = audio_file.parent / 'extracted_metadata.csv'
            if metadata_file.exists():
                df = pd.read_csv(metadata_file)
                matching_row = df[df['audio_file'] == str(audio_file)]
                if not matching_row.empty:
                    transcription = matching_row.iloc[0]['transcription']
                    print(f"   Text: {transcription}")
            
            # Display audio player
            display(Audio(audio, rate=sr))
            
            # Simple waveform plot
            plt.figure(figsize=(10, 2))
            time = np.linspace(0, duration, len(audio))
            plt.plot(time, audio)
            plt.title(f"Waveform: {audio_file.name}")
            plt.xlabel('Time (s)')
            plt.ylabel('Amplitude')
            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f"❌ Error testing {audio_file.name}: {str(e)}")
else:
    print("❌ No audio files found to test")

print("\n🎉 Audio extraction and verification complete!")

## 8. Save Extraction Report

In [ ]:
# Create comprehensive extraction report
report = {
    'extraction_date': pd.Timestamp.now().isoformat(),
    'environment': 'Google Colab',
    'torchcodec_version': torchcodec.__version__ if 'torchcodec' in globals() else 'unknown',
    'extraction_results': extraction_results,
    'verification_results': verification_results,
    'tts_datasets': tts_datasets_created,
    'total_samples': total_tts_samples,
    'output_directories': {
        'extracted_audio': str(AUDIO_OUTPUT_DIR),
        'tts_ready': str(TTS_READY_DIR)
    },
    'settings': {
        'target_sample_rate': 16000,
        'max_samples_per_dataset': MAX_SAMPLES_PER_DATASET,
        'min_duration': 0.5,
        'max_duration': 15.0
    }
}

# Save report
report_file = BASE_DIR / 'michset_audio_extraction_report.json'
with open(report_file, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f"📋 Extraction report saved: {report_file}")

# Print final summary
print("\n" + "=" * 60)
print("🚀 MICHSET AUDIO EXTRACTION COMPLETE!")
print("=" * 60)

print(f"\n📊 Final Statistics:")
print(f"   Total audio files extracted: {total_extracted}")
print(f"   TTS-ready samples: {total_tts_samples}")
print(f"   Output directory: {AUDIO_OUTPUT_DIR}")
print(f"   TTS datasets: {TTS_READY_DIR}")

print(f"\n✅ Success! Your MichSet audio files are now extracted and ready.")
print(f"\n🎯 Next: Run 005_phase3_tts_development.ipynb for TTS training!")

if total_extracted > 0:
    print(f"\n🎉 You now have real audio files for TTS training!")
    print(f"   This is a major milestone - no more dummy data!")
else:
    print(f"\n⚠️ No audio files were extracted. Check the error messages above.")
    print(f"   You may need to verify the MichSet dataset format or torchcodec installation.")